# Verbi põhjal koondkorpuse näited

### Võtab transactions x transaction_head tabelist vastava verbiga näited. 

### Sentence_id põhjal võetakse soovitud number näitelauseid koondkorpusest. 

### Tulemuseks pandase tabel ja csv, kus on verb, root sõna, lause ja veel muud infot.

Tabel on piisav, et luua uued labelstudio taskid.

In [1]:
import sqlite3
import pandas as pd
import os
from estnltk.storage.postgres import PostgresStorage, create_schema
from read_config import read_config
from sql_query import get_transaction_examples, get_koondkorpus_examples

## Configuration

In [2]:
DB_DIR = "../example_data"

PATTERN_MATCHES_DB = f"{DB_DIR}/pattern_matches.db"
TRANSACTION_DB = f"{DB_DIR}/transactions.db"
ENRICHED_TRANSACTION_DB = f"{DB_DIR}/enriched_transactions.db"

TRANSACTION_HEAD = "transaction_head"
ENRICHED_TRANSACTIONS = "transaction_row"

# verbi päringu info
KAANE = 'adit'
MAINVERB = "tulema"
COMP = ''
# kas ainult elus sõnaga verbe 
ELUS = False
# kas ainult koht sõnaga verbe
KOHT = False

# kui palju random sampleid transaction tabelist võetakse
# kui num_samples = -1 siis antakse kõik 
NUM_SAMPLES = 10

# failinimi, kuhu koondkorpuse näited salvestatakse
SAVE_FILE_NAME = "koondkorpus_examples.csv"


# juhul kui on soovi vaadata verbikäänete count tabelist verbe
TRANSACTIONS_OBL_ACTOR_LOC_COUNTS = "trans_obl_actor_loc_counts"

## Connect to db

In [3]:
con = sqlite3.connect(PATTERN_MATCHES_DB)
cur = con.cursor()
cur.execute(f'ATTACH DATABASE "{TRANSACTION_DB}" AS trans')
cur.execute(f'ATTACH DATABASE "{ENRICHED_TRANSACTION_DB}" AS entrans')

## Connect to koondkorpus

In [4]:
root = os.getcwd()
file_name = "conf_minu.ini"
if os.path.isfile(file_name):
    fname = os.path.basename(file_name).split('/')[-1]
    config = read_config(fname, file_name)
else:
    print("Could not find the specified configuration file.")
    raise SystemExit

In [5]:
source_storage = PostgresStorage(host=config["source_database"]["host"],
                          port=config["source_database"]["port"],
                          dbname=config["source_database"]["database_name"],
                          user=config["source_database"]["username"],
                          password=config["source_database"]["password"],
                          schema=config["source_database"]["work_schema"], 
                          role=config["source_database"]["role"],
                          temporary=False)

INFO:storage.py:57: connecting to host: 'postgres.keeleressursid.ee', port: '5432', dbname: 'estonian-text-corpora', user: 'koljalka'
INFO:storage.py:108: schema: 'estonian_text_corpora', temporary: False, role: 'estonian_text_corpora_read'


In [6]:
collection = source_storage['koondkorpus_sentences']

## Workflow

Võtab transaction x transaction hea dtabelist soovitud verbiga näited.

In [7]:
trans_examples = get_transaction_examples(connection=con, 
                                         #transactions_db="entrans", 
                                        transactions=ENRICHED_TRANSACTIONS, 
                                        #transhead_db="trans",
                                        transaction_head=TRANSACTION_HEAD, 
                                        mainverb=MAINVERB, 
                                        comp=COMP, 
                                        kaane=KAANE,
                                        only_elus = ELUS,
                                        only_koht = KOHT,
                                        num_samples = NUM_SAMPLES
                                         )

In [8]:
trans_examples

,head_id,sentence_id,verb,verb_compound,verb_loc,verb_form,root_deprel,kaane,root_lemma,root_form,root_loc,root_loc_rel,root_parent_loc,koht,elus
3,875,485,tulema,,6,tulen,obl,adit,kodu,koju,5,-1,None,YES,
2,731,396,tulema,,12,tuleb,obl,adit,kodu,koju,13,1,None,YES,
0,53,39,tulema,,5,tuled,obl,adit,toim,toime,4,-1,None,,
1,497,264,tulema,,12,tulnud,obl,adit,kontserdimaja,kontserdimajja,14,1,None,,


Võtab koondkorposest vastavate sentence_id-ga laused.

In [9]:
corp_examples = get_koondkorpus_examples(collection=collection, examples=trans_examples)

In [10]:
corp_examples

,head_id,sentence_id,verb,verb_compound,verb_loc,verb_form,root_deprel,kaane,root_lemma,root_form,root_loc,root_loc_rel,root_parent_loc,koht,elus,sentence
3,875,485,tulema,,6,tulen,obl,adit,kodu,koju,5,-1,None,YES,,"Niipea , kui ma koju tulen , muutub ta tujukaks ja võimukaks : “ Tee seda !"
2,731,396,tulema,,12,tuleb,obl,adit,kodu,koju,13,1,None,YES,,Milline seksuaalfantaasia tundub kõige apetiitsem : seks tundmatuga liftis ; torulukksepp tuleb koju ; jõuluvana ja snegurotška või doktor ja patsient ?
0,53,39,tulema,,5,tuled,obl,adit,toim,toime,4,-1,None,,,Kuidas sa rahaliselt toime tuled ?
1,497,264,tulema,,12,tulnud,obl,adit,kontserdimaja,kontserdimajja,14,1,None,,,"Mäe on kohtunud piletikassa juures pärnakatega , kes väidavad end olevat tulnud uude kontserdimajja juba üheksandat-kümnendat korda ."


In [11]:
corp_examples.to_csv("data/"+SAVE_FILE_NAME, sep=";", index=False, encoding="UTF-8")

#### base tabel kust saab vaadata verb counte - Pole töövooks vajalik, lisainfo
Sisaldab verb, kääne, elus_cnt, koht_cnt, distinct root count  

In [12]:
query = """SELECT * from {base_tbl} """.format(base_tbl = TRANSACTIONS_OBL_ACTOR_LOC_COUNTS)
source = pd.read_sql_query(query, con)

source["elus_div"] = source["elus_cnt"]/source["root_cnt"]
source["koht_div"] = source["koht_cnt"]/source["root_cnt"]
source["kaane2"] = ''
source.loc[source['loc_case'] == 'ill', 'kaane2'] = 'sisse'
source.loc[source['loc_case'] == 'in', 'kaane2'] = 'sees'
source.loc[source['loc_case'] == 'el', 'kaane2'] = 'seest'
source.loc[source['loc_case'] == 'all', 'kaane2'] = 'alale'
source.loc[source['loc_case'] == 'ad', 'kaane2'] = 'alal'
source.loc[source['loc_case'] == 'abl', 'kaane2'] = 'alalt'
source.loc[source['loc_case'] == 'adit', 'kaane2'] = 'sisse'

In [13]:
source

,verb,verb_compound,loc_case,elus_cnt,koht_cnt,root_cnt,elus_div,koht_div,kaane2
0,aitama,,ad,1,0,1,1.0,0.0,alal
1,ajama,,adit,1,0,1,1.0,0.0,sisse
2,algama,,el,0,0,1,0.0,0.0,seest
3,andma,,ad,0,0,1,0.0,0.0,alal
4,andma,,all,1,0,1,1.0,0.0,alale
...,...,...,...,...,...,...,...,...,...
141,võimaldama,,ad,1,0,1,1.0,0.0,alal
142,võtma,,el,0,0,1,0.0,0.0,seest
143,võtma,,in,0,0,1,0.0,0.0,sees
144,ärkama,,ad,1,0,1,1.0,0.0,alal


In [14]:
source[source["verb_compound"]!=""]

,verb,verb_compound,loc_case,elus_cnt,koht_cnt,root_cnt,elus_div,koht_div,kaane2
6,andma,alla,all,1,0,1,1.0,0.0,alale
8,astuma,üles,ad,0,0,1,0.0,0.0,alal
12,elama,välja,adit,0,0,1,0.0,0.0,sisse
13,elama,välja,in,0,0,1,0.0,0.0,sees
15,hakkama,külge,all,1,0,1,1.0,0.0,alale
19,hoidma,kokku,ad,0,0,1,0.0,0.0,alal
30,jääma,puudu,el,0,0,1,0.0,0.0,seest
44,kukkuma,alla,el,1,0,2,0.5,0.0,seest
45,kukkuma,alla,in,0,1,1,0.0,1.0,sees
46,kumama,läbi,el,0,0,1,0.0,0.0,seest


In [15]:
source_storage.close()

In [16]:
con.close()